# 03 — Del texto al registro

**Qué hace.** Extrae los registros de entrevista con dos caminos (reglas y LLM), aplica la regla de verificación literal y mide las dos capas de métricas por separado.

**Qué lee.** La caché de transcripciones (`rutas.transcripciones_dir`), `data/ground_truth/entrevistas.csv` y `data/referencias/*.txt`.

**Qué produce.** Los registros extraídos y las métricas de las dos capas en `rutas.resultados_dir`.

Llama a `src.normalizacion`, `src.extraccion_reglas`, `src.extraccion_llm`, `src.verificacion` y `src.metricas`; no reimplementa nada.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config
from src.normalizacion import normalizar_texto, palabras_a_numeros

cfg = cargar_config()
rutas = cfg["rutas"]
pd.set_option("display.max_colwidth", None)

## 1. Capa de normalización

En el papel, "seis mil quinientos" y 6500 son lo mismo. En el audio no: si la transcripción dice las palabras y el registro guarda la cifra, la verificación literal falla aunque todo esté bien. Esta capa existe solo porque la entrada es voz.

Dos funciones de `src/normalizacion.py`. Las reglas completas están en el docstring del módulo y los parámetros en `config.yaml → normalizacion`:

- **`palabras_a_numeros(texto)`** convierte los números dichos en palabras a dígitos, sin tocar el resto de la frase. Usa `text2num` y añade reglas propias donde la librería se equivoca: artículos, ordinales, apócope, "y medio", dictados y códigos.
- **`normalizar_texto(texto)`** es la regla única para comparar referencia y transcripción: números a dígitos, minúsculas, sin puntuación, espacios colapsados. Las tildes se mantienen (opción `quitar_tildes`).

In [2]:
print(json.dumps(cfg["normalizacion"], ensure_ascii=False, indent=1))

{
 "minusculas": true,
 "quitar_puntuacion": true,
 "palabras_a_numeros": true,
 "quitar_tildes": false,
 "colapsar_espacios": true,
 "umbral_numeros_pequenos": 2,
 "articulos": [
  "un",
  "una",
  "uno",
  "unos",
  "unas"
 ],
 "alias": {
  "veintiún": "veintiuno",
  "veintiun": "veintiuno",
  "veintidos": "veintidós",
  "veintitres": "veintitrés",
  "veintiseis": "veintiséis"
 },
 "palabras_protegidas": [
  "por ciento",
  "primer",
  "primero",
  "primera",
  "segundo",
  "segunda",
  "tercer",
  "tercero",
  "tercera",
  "cuarto",
  "cuarta",
  "quinto",
  "quinta",
  "sexto",
  "sexta",
  "séptimo",
  "séptima",
  "octavo",
  "octava",
  "noveno",
  "novena",
  "décimo",
  "décima"
 ],
 "unidades_y_medio": [
  "año",
  "años",
  "mes",
  "meses",
  "hora",
  "horas"
 ],
 "min_digitos_dictado": 3,
 "codigos": [
  "LEÑA5"
 ]
}


### 1.1 Casos de `palabras_a_numeros`

Cada caso sale de los guiones. Las filas con `limitacion = True` son casos sin una buena solución en esta capa: se muestran con su salida ideal (`esperado`) y con la que de verdad se obtiene (`queda`), en vez de esconderlos. Los casos marcados *(hallado en …)* no estaban en la lista inicial: aparecieron al aplicar la función a las referencias completas (sección 1.4).

In [3]:
casos = [
    # grupo, entrada, esperado, motivo
    ("cifras", "seis mil quinientos soles", "6500 soles", "pretensión de E1"),
    ("cifras", "mil ochocientos dólares", "1800 dólares", "pretensión de E2"),
    ("cifras", "nueve mil quinientos", "9500", "pretensión ideal de E3"),
    ("direcciones", "Calle Los Cedros doscientos cuarenta y cinco", "Calle Los Cedros 245", "L1"),
    ("direcciones", "Jirón Los Halcones mil treinta y cuatro", "Jirón Los Halcones 1034", "L3"),
    ("direcciones", "Avenida Los Laureles dos mil doscientos setenta y uno", "Avenida Los Laureles 2271", "L5"),
    ("años", "desde dos mil veintitrés, o sea, tres años", "desde 2023, o sea, 3 años",
     "el año queda como cifra; pasarlo a duración es trabajo de la extracción"),
    ("años", "empecé como tester manual en dos mil veinte", "empecé como tester manual en 2020", "E5"),
    ("apócope", "su total es ciento veintiún soles", "su total es 121 soles",
     "text2num no conoce 'veintiún': alias a 'veintiuno'"),
    ("apócope", "el total es ciento treinta y un soles", "el total es 131 soles", "'y un' dentro del compuesto"),
    ("tildes", "dos mollejitas, a dieciséis cada una", "2 mollejitas, a 16 cada una", "L4"),
    ("tildes", "veintitrés", "23", "con tilde"),
    ("tildes", "veintitres", "23", "si el ASR omite la tilde: alias"),
    ("teléfono", "nueve ocho siete, seis cinco cuatro, tres dos uno", "987, 654, 321",
     "dictado por grupos: cada grupo se une, las comas separan"),
    ("artículos", "un pollo entero", "un pollo entero", "artículo: la cantidad 1 la decide la extracción"),
    ("artículos", "una jarra de chicha morada", "una jarra de chicha morada", "artículo"),
    ("artículos", "un medio pollo", "un medio pollo", "artículo + tamaño"),
    ("artículos", "dos jarras de chicha morada", "2 jarras de chicha morada", "desde 'dos' sí se convierte"),
    ("artículos", "casi un quince por ciento", "casi un 15 por ciento", "hallado en E2: text2num daba '1 15'"),
    ("artículos", "le llega en unos cuarenta minutos", "le llega en unos 40 minutos", "hallado en L2: daba '1 40'"),
    ("artículos", "a veinte soles cada uno, y tres papas", "a 20 soles cada uno, y 3 papas",
     "hallado en L3: daba 'cada 1, y 3'"),
    ("artículos", "de treinta y seis y una ensalada clásica", "de 36 y una ensalada clásica",
     "hallado en L4: tras 'seis y' es artículo, tras 'treinta y' sería número"),
    ("medio", "medio pollo", "medio pollo", "'medio' es un tamaño"),
    ("medio", "una papa familiar", "una papa familiar", "artículo + tamaño"),
    ("medio", "una gaseosa de litro y medio", "una gaseosa de litro y medio", "tamaño de la carta, no tiempo"),
    ("medio", "cuatro años y medio", "4.5 años", "número + unidad de tiempo + 'y medio'"),
    ("medio", "un año y medio", "1.5 año", "E4: el artículo también vale como 1 aquí"),
    ("ordinales", "tres cuartos de pollo", "3 cuartos de pollo", "text2num daba '3 4ᵒˢ': ordinal protegido"),
    ("ordinales", "primero de diciembre", "primero de diciembre", "la fecha la interpreta la extracción"),
    ("ordinales", "la segunda semana de enero", "la segunda semana de enero", "text2num daba '2ª'"),
    ("ordinales", "cien por ciento", "100 por ciento", "text2num daba '100 por 100'"),
    ("código", "LEÑA cinco", "LEÑA5", "código de promoción unido"),
    ("código", "Leña 5", "LEÑA5", "si el ASR ya escribe la cifra"),
    ("código", "La Leña Dorada", "La Leña Dorada", "sin número detrás no es el código"),
    ("autocorrección", "mil cuarenta y tres… no, perdón, me equivoqué, es mil treinta y cuatro",
     "1043… no, perdón, me equivoqué, es 1034", "se convierten los dos; el final lo elige la extracción"),
]

limitaciones = [
    # grupo, entrada, esperado (ideal), queda (lo que hace), motivo
    ("rango elíptico", "entre nueve y diez mil soles", "entre 9000 y 10000 soles", "entre 9 y 10000 soles",
     "el 'mil' se aplica a los dos extremos; resolverlo es interpretar el rango, no convertir palabras. "
     "En E3 no afecta: vale el ideal 9500"),
    ("medio sin número", "llevo año y medio", "llevo 1.5 año", "llevo año y medio",
     "la regla exige número o artículo delante; no aparece en el corpus"),
    ("forma literaria", "mil y una noches", "1001 noches", "1000 y una noches",
     "tras 'mil y' la regla de artículos aparta 'una'; no aparece en el corpus"),
]

tabla = pd.DataFrame(
    [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e),
      "limitacion": False, "motivo": m} for g, e, esp, m in casos]
    + [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e),
        "limitacion": True, "motivo": m} for g, e, esp, _queda, m in limitaciones]
)
tabla.insert(4, "pasa", tabla.obtenido == tabla.esperado)
print(f"{tabla.pasa.sum()} de {len(tabla)} casos pasan · {tabla.limitacion.sum()} limitaciones documentadas")
tabla

35 de 38 casos pasan · 3 limitaciones documentadas


,grupo,entrada,esperado,obtenido,pasa,limitacion,motivo
0,cifras,seis mil quinientos soles,6500 soles,6500 soles,True,False,pretensión de E1
1,cifras,mil ochocientos dólares,1800 dólares,1800 dólares,True,False,pretensión de E2
2,cifras,nueve mil quinientos,9500,9500,True,False,pretensión ideal de E3
3,direcciones,Calle Los Cedros doscientos cuarenta y cinco,Calle Los Cedros 245,Calle Los Cedros 245,True,False,L1
4,direcciones,Jirón Los Halcones mil treinta y cuatro,Jirón Los Halcones 1034,Jirón Los Halcones 1034,True,False,L3
5,direcciones,Avenida Los Laureles dos mil doscientos setenta y uno,Avenida Los Laureles 2271,Avenida Los Laureles 2271,True,False,L5
6,años,"desde dos mil veintitrés, o sea, tres años","desde 2023, o sea, 3 años","desde 2023, o sea, 3 años",True,False,el año queda como cifra; pasarlo a duración es trabajo de la extracción
7,años,empecé como tester manual en dos mil veinte,empecé como tester manual en 2020,empecé como tester manual en 2020,True,False,E5
8,apócope,su total es ciento veintiún soles,su total es 121 soles,su total es 121 soles,True,False,text2num no conoce 'veintiún': alias a 'veintiuno'
9,apócope,el total es ciento treinta y un soles,el total es 131 soles,el total es 131 soles,True,False,'y un' dentro del compuesto


### 1.2 Asserts

El notebook falla si un caso deja de dar su salida. Para las limitaciones se comprueba que siguen comportándose como está documentado: si una cambia (para bien o para mal), hay que enterarse.

In [4]:
for grupo, entrada, esperado, motivo in casos:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == esperado, f"[{grupo}] {entrada!r}: se esperaba {esperado!r}, salió {obtenido!r}"

for grupo, entrada, esperado, queda, motivo in limitaciones:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == queda, f"[{grupo}] la limitación cambió: {entrada!r} ahora da {obtenido!r}"
    assert obtenido != esperado, f"[{grupo}] ya no es limitación: muévelo a casos"

print(f"OK: {len(casos)} casos y {len(limitaciones)} limitaciones se comportan como está documentado")

OK: 35 casos y 3 limitaciones se comportan como está documentado


### 1.3 `normalizar_texto`: la referencia y la transcripción deben quedar iguales

La referencia escribe los números en palabras y Whisper suele escribirlos en cifras, con separador de miles o con `%`. Si la normalización no los iguala, el WER cuenta como error algo que el modelo oyó bien. Cada par es *(como está en el guion, como podría escribirlo el ASR)*.

In [5]:
pares = [
    # referencia, variante ASR, deben_coincidir, motivo
    ("Mi pretensión sería de seis mil quinientos soles.", "mi pretensión sería de 6.500 soles", True,
     "separador de miles"),
    ("cuatro años y medio", "4,5 años", True, "coma decimal"),
    ("casi un quince por ciento", "casi un 15%", True, "'%' -> 'por ciento'"),
    ("¿Cuál es tu pretensión salarial?", "Cuál es tu pretensión salarial", True, "puntuación y mayúsculas"),
    ("Next.js", "next.js", True, "el punto entre letras es puntuación"),
    ("nueve ocho siete, seis cinco cuatro, tres dos uno", "987 654 321", True, "dictado por grupos"),
    ("Ah, tengo un código de la aplicación: LEÑA cinco.", "Ah, tengo un código de la aplicación: Leña 5.", True,
     "código"),
    ("Está bien.", "esta bien", False, "decisión: las tildes se mantienen (quitar_tildes: false)"),
    ("nueve ocho siete, seis cinco cuatro, tres dos uno", "987654321", False,
     "limitación: un teléfono sin espacios no se puede volver a partir en grupos"),
]

comparacion = pd.DataFrame(
    [{"referencia": r, "variante_asr": v, "ref_normalizada": normalizar_texto(r),
      "asr_normalizada": normalizar_texto(v), "deben_coincidir": d, "motivo": m} for r, v, d, m in pares]
)
comparacion.insert(4, "coinciden", comparacion.ref_normalizada == comparacion.asr_normalizada)
comparacion

,referencia,variante_asr,ref_normalizada,asr_normalizada,coinciden,deben_coincidir,motivo
0,Mi pretensión sería de seis mil quinientos soles.,mi pretensión sería de 6.500 soles,mi pretensión sería de 6500 soles,mi pretensión sería de 6500 soles,True,True,separador de miles
1,cuatro años y medio,"4,5 años",4.5 años,4.5 años,True,True,coma decimal
2,casi un quince por ciento,casi un 15%,casi un 15 por ciento,casi un 15 por ciento,True,True,'%' -> 'por ciento'
3,¿Cuál es tu pretensión salarial?,Cuál es tu pretensión salarial,cuál es tu pretensión salarial,cuál es tu pretensión salarial,True,True,puntuación y mayúsculas
4,Next.js,next.js,next js,next js,True,True,el punto entre letras es puntuación
5,"nueve ocho siete, seis cinco cuatro, tres dos uno",987 654 321,987 654 321,987 654 321,True,True,dictado por grupos
6,"Ah, tengo un código de la aplicación: LEÑA cinco.","Ah, tengo un código de la aplicación: Leña 5.",ah tengo un código de la aplicación leña5,ah tengo un código de la aplicación leña5,True,True,código
7,Está bien.,esta bien,está bien,esta bien,False,False,decisión: las tildes se mantienen (quitar_tildes: false)
8,"nueve ocho siete, seis cinco cuatro, tres dos uno",987654321,987 654 321,987654321,False,False,limitación: un teléfono sin espacios no se puede volver a partir en grupos


In [6]:
for r, v, d, m in pares:
    assert (normalizar_texto(r) == normalizar_texto(v)) == d, f"{r!r} / {v!r}: {m}"

assert normalizar_texto("Mi pretensión sería de seis mil quinientos soles.") == "mi pretensión sería de 6500 soles"
assert normalizar_texto("4,5 años") == "4.5 años"
assert normalizar_texto("1.5 litros") == "1.5 litros"
assert normalizar_texto("1.000.000") == "1000000"

# quitar_tildes es una opción de config.yaml; se prueba sin editar el archivo pasando otras reglas.
con_tildes_fuera = {**cfg["normalizacion"], "quitar_tildes": True}
assert normalizar_texto("Está bien.", con_tildes_fuera) == normalizar_texto("esta bien", con_tildes_fuera)
assert normalizar_texto("La Leña Dorada, pingüino", con_tildes_fuera) == "la leña dorada pinguino"
print("OK: normalizar_texto")

OK: normalizar_texto


### 1.4 Los números de cada referencia contra el ground truth

`palabras_a_numeros` se aplica a los 10 textos de `data/referencias/`. Por cada clip se listan, en orden, todos los números sueltos que aparecen tras convertir (no cuenta el `1` de "C1" ni el `5` de "LEÑA5"). Así se puede revisar a ojo que no haya conversiones de más; con esta revisión se encontraron los cuatro casos de artículos de la tabla 1.1.

In [7]:
referencias = {
    f.stem: f.read_text(encoding="utf-8")
    for f in sorted(rutas["referencias_dir"].glob(cfg["archivos"]["patron_referencia"].format(clip="*")))
}
numeros = pd.DataFrame(
    [{"clip": clip, "cantidad": len(n := re.findall(r"\b\d+(?:\.\d+)?\b", palabras_a_numeros(texto))),
      "numeros_encontrados": " · ".join(n)} for clip, texto in referencias.items()]
)
numeros

,clip,cantidad,numeros_encontrados
0,e1,11,2 · 5 · 4.5 · 6 · 2 · 2 · 2 · 5800 · 6500 · 2 · 6
1,e2,7,2 · 2023 · 3 · 2 · 15 · 6 · 1800
2,e3,9,9 · 10000 · 9500 · 3 · 8 · 9 · 9 · 8 · 2
3,e4,7,100 · 1.5 · 4200 · 4800 · 6 · 2 · 3
4,e5,9,2 · 4 · 2 · 11 · 6200 · 7000 · 2020 · 6 · 2
5,l1,17,68 · 14 · 2 · 2 · 12 · 10 · 245 · 302 · 245 · 302 · 987 · 654 · 321 · 2 · 96 · 40 · 50
6,l2,20,2 · 2 · 36 · 2 · 72 · 68 · 2 · 2 · 3 · 18 · 9 · 5 · 1580 · 2 · 1580 · 2 · 100 · 95 · 95 · 40
7,l3,25,3 · 3 · 3 · 20 · 3 · 7 · 3 · 3 · 4 · 2 · 2 · 9 · 1043 · 5 · 1034 · 1034 · 5 · 1034 · 5 · 3 · 3 · 3 · 2 · 121 · 45
8,l4,16,36 · 10 · 2 · 2 · 16 · 2 · 2 · 5 · 5 · 418 · 702 · 418 · 702 · 2 · 85 · 35
9,l5,15,2 · 2 · 2 · 2 · 2 · 2 · 10 · 2271 · 504 · 2271 · 504 · 2 · 2 · 131 · 131


Ahora, cada valor numérico del ground truth: ¿aparece literalmente en la referencia después de normalizar? Se busca como palabra completa (`"8"` no coincide dentro de `"1800"`). Es un adelanto de la regla de verificación, que implementará `src/verificacion.py`.

Se revisan los años de experiencia y la pretensión (Task 1), y el total dicho, el descuento, el precio de cada ítem y los números de la dirección (Task 2). Las cantidades no se revisan: una cifra como `2` aparece en casi cualquier llamada, así que encontrarla no demuestra nada.

In [8]:
entrevistas = pd.read_csv(rutas["ground_truth_entrevistas"])
llamadas = pd.read_csv(rutas["ground_truth_llamadas"])


def como_token(x):
    return f"{float(x):g}"


valores = []
for _, f in entrevistas.iterrows():
    valores += [(f["clip"], "anios_experiencia", f["anios_experiencia"]),
                (f["clip"], "pretension_salarial", f["pretension_salarial"])]
for _, f in llamadas.iterrows():
    valores.append((f["clip"], "total_dicho", f["total_dicho"]))
    if f["descuento"]:
        valores.append((f["clip"], "descuento", f["descuento"]))
    for item in json.loads(f["items"]):
        valores.append((f["clip"], f"precio {item['producto']} ({item['tamano']})", item["precio_unitario"]))
    for n in re.findall(r"\d+", f["direccion_entrega"]):
        valores.append((f["clip"], "direccion_entrega", n))

tokens = {clip: set(normalizar_texto(texto).split()) for clip, texto in referencias.items()}
literal = pd.DataFrame(
    [{"clip": c, "campo": campo, "valor_gt": como_token(v), "aparece": como_token(v) in tokens[c]}
     for c, campo, v in valores]
)
literal.groupby("clip").aparece.agg(revisados="count", aparecen="sum").T

clip,e1,e2,e3,e4,e5,l1,l2,l3,l4,l5
revisados,2,2,2,2,2,7,6,7,9,8
aparecen,2,2,2,1,2,7,6,7,8,4


Los valores del ground truth que **no** aparecen literalmente:

In [9]:
literal[~literal.aparece]

,clip,campo,valor_gt,aparece
6,e4,anios_experiencia,2.5,False
35,l4,precio anticuchos (porción),18,False
41,l5,precio pollo a la brasa (entero),68,False
42,l5,precio pollo a la brasa (medio),36,False
43,l5,precio papas fritas (familiar),14,False
44,l5,precio gaseosa (1.5 litros),9,False


**Qué dicen estos ausentes**

- **E4 · anios_experiencia = 2.5.** Nunca se dice: el candidato cuenta "un año y medio" (1.5) y "otro año". La normalización produce `1.5`, no `2.5`. La extracción puede acertar sumando, pero la regla de verificación debe dejar el campo `sin_verificar`. Esta es la trampa que se plantó en E4.
- **L5 · todos los precios.** El vendedor nunca dicta los precios en esta llamada, y en L4 no dice el de los anticuchos. El precio unitario sale de la carta, no del audio. La verificación de `items` tiene que contrastar los precios con `carta.csv`, porque contra la transcripción nunca podría verificarlos.

**Lo que esta prueba no garantiza.** Que un número aparezca no prueba que ocupe el lugar correcto. En E3 el `8` de la experiencia está, pero también está `9` ("casi nueve"). En L3 aparecen `1043` y `1034`. Elegir cuál es el valor es trabajo de la extracción; esta capa solo garantiza que, si se dijo, se puede encontrar.

## 2. Cargar transcripciones y ground truth

Desde la caché; aquí no se transcribe.

*Pendiente.*

## 3. Extracción por reglas

Expresiones regulares por campo.

*Pendiente.*

## 4. Extracción por LLM

JSON estructurado con el prompt de `config.yaml`. Cada valor viene con su evidencia literal.

*Pendiente.*

## 5. Verificación

Todo valor que no existe literalmente en la transcripción queda `sin_verificar`. Caso esperado: E4, `anios_experiencia`.

*Pendiente.*

## 6. Capa de transcripción

WER y CER por clip.

*Pendiente.*

## 7. Capa de registro

Exactitud por campo, macro F1 en `nivel_ingles` y error absoluto en los campos numéricos, con intervalos de confianza.

*Pendiente.*

## 8. ¿El WER predice si el registro sirve?

Un clip con WER alto y registro perfecto, y uno con WER bajo y registro roto.

*Pendiente.*

## 9. Reglas contra LLM

Comparación en los mismos clips.

*Pendiente.*